# 1. Build predicted 3Di libraries

**Paper:** Predicted 3Di Foldseek databases (ESM3-3Di, ESM3-LoRA, ProstT5, SaProt).

This notebook does **not** run the translation models. Homology search uses **AA→3Di only**. Bidirectional translation accuracy is `2b_translation_accuracy.ipynb`.

| | Path |
|--|------|
| Input | `0_prepare_scope40.ipynb` products: `work/GT_fasta/`, `work/DB/{foldseek,mmseqs}_DB/`, `work/labels/scop_lookup.tsv`, `bin/` |
| | User AA→3Di FASTA in `work/aa2di_fasta/` (headers must match `DB_aa.fasta`) |
| Output | `work/DB/{ESM3,ESM3_LoRA,ProstT5,SaProt}_DB/` |

Required filenames (copy after running models on `work/GT_fasta/DB_aa.fasta`):

| Method | `work/aa2di_fasta/` |
|--------|---------------------|
| ESM3-3Di | `DB_ESM3_aa2di.fasta` |
| ESM3-LoRA | `DB_ESM3_LoRA_aa2di.fasta` |
| ProstT5 (translate) | `DB_ProstT5_translate_aa2di.fasta` |
| SaProt | `DB_SaProt_aa2di.fasta` |

Optional for 2b (not used here): `work/di2aa_fasta/DB_*_di2aa.fasta` against `GT_fasta/DB_di.fasta`.

**Node:** login / light CPU. ~10 min.

**Next:** `2a_remote_homology.ipynb` (compute node) and/or `2b_translation_accuracy.ipynb`.


## Environment


In [ ]:
NOTEBOOK_NAME = "1_build_predicted_dbs.ipynb"

import os
import platform
import subprocess
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
if not (cwd / NOTEBOOK_NAME).is_file():
    raise SystemExit(
        f"Start this notebook from the project root (cwd must contain {NOTEBOOK_NAME}). "
        f"Current cwd: {cwd}"
    )

CONDA_ENV = "ESM3_3Di_5090"
print("notebook:", NOTEBOOK_NAME)
print("cwd:", cwd)
print("python:", sys.executable)
print("version:", sys.version.split()[0])
print("platform:", platform.platform())
print("CONDA_DEFAULT_ENV:", os.environ.get("CONDA_DEFAULT_ENV", "(unset)"))

if CONDA_ENV not in sys.executable:
    expected = Path.home() / ".conda" / "envs" / CONDA_ENV / "bin" / "python"
    raise SystemExit(
        f"Kernel is not {CONDA_ENV} (current: {sys.executable}). "
        f"Select kernel {CONDA_ENV} and Restart. Do not pip into miniforge3 python3.12. "
        f"Expected: {expected}"
    )


def _bin_version(name: str) -> str:
    path = cwd / "bin" / name
    if not path.is_file():
        return "(not installed yet; run 0_prepare_scope40.ipynb)"
    try:
        proc = subprocess.run([str(path), "version"], capture_output=True, text=True, check=False)
        lines = (proc.stdout or proc.stderr or "").strip().splitlines()
        return lines[0] if lines else "(unknown)"
    except OSError as exc:
        return f"(failed: {exc})"


print("foldseek:", _bin_version("foldseek"))
print("mmseqs:", _bin_version("mmseqs"))


## Configuration

本格在五本 notebook 中**字节级相同**。改方法表、搜索参数或 URL 时：只改 `0_prepare_scope40.ipynb` 这一格，再整格复制到另外四本。发布前可用 checksum 核对五本是否一致。


In [ ]:
# =============================================================================
# Configuration — copy this entire cell into all five notebooks.
# Change methods or search parameters here in 0_prepare_scope40.ipynb, then
# paste the same cell into 1_build / 2a / 2b / 3_figures.
# =============================================================================
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
HOME = ROOT.parent

CONDA_ENV = "ESM3_3Di_5090"
FOLDSEEK_VERSION = "10-941cd33"
MMSEQS_VERSION = "18-8cc5c"

TEMP = ROOT / "tmp"
WORK_DIR = ROOT / "work"
BIN_DIR = ROOT / "bin"
WORK_TMP_DIR = WORK_DIR / "tmp"

GT_FASTA_DIR = WORK_DIR / "GT_fasta"
AA_FASTA = GT_FASTA_DIR / "DB_aa.fasta"
GT_DI_FASTA = GT_FASTA_DIR / "DB_di.fasta"

AA2DI_FASTA_DIR = WORK_DIR / "aa2di_fasta"
DI2AA_FASTA_DIR = WORK_DIR / "di2aa_fasta"

DBS_DIR = WORK_DIR / "DB"
FOLDSEEK_GT_DIR = DBS_DIR / "foldseek_DB"
MMSEQS_GT_DIR = DBS_DIR / "mmseqs_DB"

LABEL_DIR = WORK_DIR / "labels"
SCOP_LOOKUP = LABEL_DIR / "scop_lookup.tsv"
LEGACY_LABEL_DIR = WORK_DIR / "lable"

ALN_DIR = WORK_DIR / "aln"
METRICS_DIR = WORK_DIR / "metrics"
FIGURES_DIR = WORK_DIR / "figures"
TRANSLATION_METRICS_DIR = METRICS_DIR / "translation"
WORK_BUNDLE = WORK_DIR / "scope40_work_bundle.tar.gz"

FOLDSEEK_BIN = BIN_DIR / "foldseek"
MMSEQS_BIN = BIN_DIR / "mmseqs"

FOLDSEEK_URL = (
    "https://github.com/steineggerlab/foldseek/releases/download/"
    f"{FOLDSEEK_VERSION}/foldseek-linux-avx2.tar.gz"
)
MMSEQS_URL = (
    "https://github.com/soedinglab/MMseqs2/releases/download/"
    f"{MMSEQS_VERSION}/mmseqs-linux-avx2.tar.gz"
)
FOLDSEEK_TMP_DIR = TEMP / "foldseek"
MMSEQS_TMP_DIR = TEMP / "mmseqs"
FOLDSEEK_TARBALL = TEMP / "foldseek-linux-avx2.tar.gz"
MMSEQS_TARBALL = TEMP / "mmseqs-linux-avx2.tar.gz"

SCOP_CLA_NAME = "dir.cla.scope.2.08-stable.txt"
SCOP_DES_NAME = "dir.des.scope.2.08-stable.txt"
SOURCE_ARCHIVE_NAME = "pdbstyle-sel-gs-bib-40-2.08.tgz"
SCOP_CLA_FALLBACK = HOME / "SCOPE" / SCOP_CLA_NAME
HF_BASE = "https://huggingface.co/datasets/caijihuize/scope40_pdbstyle/resolve/main"

# Foldseek / predicted 3Di: aligned with new_scope40 easy-search
EASY_SEARCH_PARAMS = {
    "sensitivity": 9.5,
    "max_seqs": 2000,
    "evalue": 10.0,
    "threads": 64,
}
# MMseqs2: aligned with foldseek-analysis/scopbenchmark/scripts/runMMseqs.sh
MMSEQS_SEARCH_PARAMS = {
    "sensitivity": 7.5,
    "max_seqs": 2000,
    "evalue": 10000,
    "threads": 64,
    "add_backtrace": True,
}
PREPARE_THREADS = 16
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# Homology search methods. protocol must not be mixed as one AUC.
METHODS: list[dict] = [
    {"name": "Foldseek (AA+3Di)", "key": "foldseek", "engine": "foldseek", "aa2di": None, "protocol": "hitlist"},
    {"name": "MMseqs2", "key": "mmseqs", "engine": "mmseqs", "aa2di": None, "protocol": "catalog"},
    {"name": "ESM3-3Di", "key": "ESM3", "engine": "foldseek", "aa2di": "DB_ESM3_aa2di.fasta", "protocol": "hitlist"},
    {"name": "ESM3-LoRA", "key": "ESM3_LoRA", "engine": "foldseek", "aa2di": "DB_ESM3_LoRA_aa2di.fasta", "protocol": "hitlist"},
    {"name": "ProstT5 (translate)", "key": "ProstT5", "engine": "foldseek", "aa2di": "DB_ProstT5_translate_aa2di.fasta", "protocol": "hitlist"},
    {"name": "SaProt", "key": "SaProt", "engine": "foldseek", "aa2di": "DB_SaProt_aa2di.fasta", "protocol": "hitlist"},
]
# Translation accuracy (bidirectional). Homology search uses aa2di only.
TRANSLATION_METHODS: list[dict] = [
    {"name": "ESM3-3Di", "key": "ESM3", "aa2di": "DB_ESM3_aa2di.fasta", "di2aa": "DB_ESM3_di2aa.fasta"},
    {"name": "ESM3-LoRA", "key": "ESM3_LoRA", "aa2di": "DB_ESM3_LoRA_aa2di.fasta", "di2aa": "DB_ESM3_LoRA_di2aa.fasta"},
    {"name": "ProstT5 (translate)", "key": "ProstT5", "aa2di": "DB_ProstT5_translate_aa2di.fasta", "di2aa": "DB_ProstT5_translate_di2aa.fasta"},
    {"name": "SaProt", "key": "SaProt", "aa2di": "DB_SaProt_aa2di.fasta", "di2aa": "DB_SaProt_di2aa.fasta"},
]
PALETTE = {
    "Foldseek (AA+3Di)": "#2b5c8f",
    "MMseqs2": "#666666",
    "ESM3-3Di": "#d95f02",
    "ESM3-LoRA": "#1b9e77",
    "ProstT5 (translate)": "#7570b3",
    "SaProt": "#e7298a",
}


def run_cmd(argv: list[str]) -> None:
    """Print then run an external command. Logs are supplementary material."""
    print("[CMD]", " ".join(str(x) for x in argv), flush=True)
    subprocess.run([str(x) for x in argv], check=True)


def require_file(path: Path, hint: str) -> Path:
    """Fail with a pointer to the upstream notebook if a required file is missing."""
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}\n{hint}")
    return path


def meta_path(output: Path) -> Path:
    return output.with_name(output.name + ".meta.json")


def skip_if_exists(output: Path, payload: dict | None = None, skip_existing: bool = True) -> bool:
    """Skip when output exists. If payload is given, require a matching .meta.json.

    MMseqs TSV without a fingerprint is treated as stale (catalog protocol change).
    Other outputs without a fingerprint are kept; set SKIP_EXISTING=False to force.
    """
    if not skip_existing:
        return False
    if not output.is_file() or output.stat().st_size == 0:
        return False
    if payload is None:
        return True
    meta = meta_path(output)
    if not meta.is_file():
        if payload.get("engine") == "mmseqs":
            print(f"[rerun] {output.name}: no fingerprint; MMseqs catalog params need a fresh search")
            return False
        print(f"[warn] {output.name}: no fingerprint; keeping existing file (set SKIP_EXISTING=False to re-run)")
        return True
    try:
        stored = json.loads(meta.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return False
    if stored != payload:
        print(f"[rerun] {output.name}: Configuration changed")
        return False
    return True


def write_meta(output: Path, payload: dict) -> None:
    meta_path(output).write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def search_params_payload(engine: str, threads: int | None = None) -> dict:
    params = dict(MMSEQS_SEARCH_PARAMS if engine == "mmseqs" else EASY_SEARCH_PARAMS)
    params["engine"] = engine
    if threads is not None:
        params["threads"] = int(threads)
    return params


def method_by_key(method_key: str) -> dict:
    for row in METHODS:
        if row["key"] == method_key:
            return row
    raise KeyError(f"Unknown method_key: {method_key}")


def predicted_methods() -> list[dict]:
    return [row for row in METHODS if row["aa2di"] is not None]


def db_prefix(method_key: str) -> Path:
    return DBS_DIR / f"{method_key}_DB" / "DB"


def aln_tsv(method_key: str) -> Path:
    return ALN_DIR / f"{method_key}_easy.tsv"


def aln_tmp_dir(method_key: str) -> Path:
    return WORK_TMP_DIR / f"easy_{method_key}"


def metric_prefix(method_key: str) -> Path:
    return METRICS_DIR / f"{method_key}_easy"


def translation_per_seq_path(task: str, method_key: str) -> Path:
    return TRANSLATION_METRICS_DIR / f"{task}_{method_key}_per_seq.tsv"


def translation_summary_path(task: str) -> Path:
    return TRANSLATION_METRICS_DIR / f"{task}_summary.csv"


def scop_cla_path() -> Path:
    for path in (TEMP / SCOP_CLA_NAME, SCOP_CLA_FALLBACK):
        if path.is_file():
            return path
    return TEMP / SCOP_CLA_NAME


def work_ready() -> bool:
    return (
        (FOLDSEEK_GT_DIR / "DB").is_file()
        and (MMSEQS_GT_DIR / "DB").is_file()
        and AA_FASTA.is_file()
        and GT_DI_FASTA.is_file()
        and SCOP_LOOKUP.is_file()
    )


def ensure_work_dirs() -> None:
    for directory in (
        TEMP, BIN_DIR, GT_FASTA_DIR, AA2DI_FASTA_DIR, DI2AA_FASTA_DIR, DBS_DIR,
        LABEL_DIR, ALN_DIR, METRICS_DIR, TRANSLATION_METRICS_DIR, FIGURES_DIR, WORK_TMP_DIR,
    ):
        directory.mkdir(parents=True, exist_ok=True)
    legacy = LEGACY_LABEL_DIR / "scop_lookup.tsv"
    if not SCOP_LOOKUP.is_file() and legacy.is_file():
        shutil.copy2(legacy, SCOP_LOOKUP)
        print(f"[ok] migrated {legacy} -> {SCOP_LOOKUP}")


def cleanup_tmp(*, also_work_tmp: bool = True) -> None:
    """Remove tmp/ and work/tmp/ only. Keep work/ products and bin/."""
    targets = [TEMP]
    if also_work_tmp:
        targets.append(WORK_TMP_DIR)
    for path in targets:
        if path.exists():
            shutil.rmtree(path)
            print(f"[ok] cleaned {path}")
        else:
            print(f"[skip] {path} (absent)")


print("ROOT:", ROOT)
print("conda env:", CONDA_ENV)
print("Foldseek:", FOLDSEEK_VERSION, "MMseqs:", MMSEQS_VERSION)
print("METHODS:", [(m["key"], m["engine"], m["protocol"]) for m in METHODS])


## Run flags


In [ ]:
SKIP_EXISTING = True
ONLY_METHODS = None  # e.g. ["ESM3", "ProstT5"]; None = all predicted methods

ensure_work_dirs()
print("GT_fasta:", GT_FASTA_DIR)
print("aa2di_fasta:", AA2DI_FASTA_DIR)
print("di2aa_fasta (for 2b, not this notebook):", DI2AA_FASTA_DIR)
print("predicted FASTA:", [m["aa2di"] for m in predicted_methods()])
print(f"SKIP_EXISTING={SKIP_EXISTING}  ONLY_METHODS={ONLY_METHODS}")


## Helpers


In [ ]:
import tempfile

def selected_predicted() -> list[dict]:
    rows = predicted_methods()
    if ONLY_METHODS is None:
        return rows
    return [m for m in rows if m["key"] in ONLY_METHODS]


def check_prepare_products() -> None:
    """Require 0_prepare_scope40.ipynb outputs. No legacy runtime download."""
    require_file(FOLDSEEK_BIN, hint="Run 0_prepare_scope40.ipynb (Step 1).")
    require_file(MMSEQS_BIN, hint="Run 0_prepare_scope40.ipynb (Step 1).")
    require_file(FOLDSEEK_GT_DIR / "DB", hint="Run 0_prepare_scope40.ipynb.")
    require_file(MMSEQS_GT_DIR / "DB", hint="Run 0_prepare_scope40.ipynb.")
    require_file(AA_FASTA, hint="Run 0_prepare_scope40.ipynb.")
    require_file(GT_DI_FASTA, hint="Run 0_prepare_scope40.ipynb.")
    require_file(SCOP_LOOKUP, hint="Run 0_prepare_scope40.ipynb.")
    print("[ok] 0_prepare products are present")


def check_aa2di_fastas() -> None:
    missing = []
    for method in selected_predicted():
        path = AA2DI_FASTA_DIR / method["aa2di"]
        if path.is_file():
            print(f"[ok] {method['name']}: {path}")
        else:
            print(f"[missing] {method['name']}: {path}")
            missing.append(method["aa2di"])
    if missing:
        listing = "\n".join(f"  - {name}" for name in missing)
        raise FileNotFoundError(
            f"Copy AA→3Di predicted FASTA into {AA2DI_FASTA_DIR}/:\n{listing}\n"
            f"Reference input: {AA_FASTA}"
        )


def _seqio():
    try:
        from Bio import SeqIO
    except ImportError as exc:
        raise SystemExit("Need Biopython in ESM3_3Di_5090: conda install biopython") from exc
    return SeqIO


def _read_fasta_dict(path: Path) -> dict[str, str]:
    SeqIO = _seqio()
    out: dict[str, str] = {}
    for record in SeqIO.parse(path, "fasta"):
        out[record.id] = str(record.seq)
    return out


print("helpers ready")


## Step 1 — Check prepare products and predicted FASTA

Stops if `0_prepare_scope40.ipynb` was not run, or if a required aa2di FASTA is missing.


In [ ]:
check_prepare_products()
check_aa2di_fastas()


## Step 2 — tsv2db for each predicted 3Di FASTA

Pair `DB_aa.fasta` with the predicted 3Di strings (uppercased). Write AA / 3Di / header TSV, then `foldseek tsv2db`.


In [ ]:
def build_tsvs(aa_fasta: Path, di_fasta: Path, tmp_dir: Path) -> None:
    SeqIO = _seqio()
    sequences_aa = _read_fasta_dict(aa_fasta)
    sequences_3di: dict[str, str] = {}
    for record in SeqIO.parse(di_fasta, "fasta"):
        if record.id not in sequences_aa:
            print(f"[warn] ignoring 3Di id {record.id} (not in AA FASTA)")
        else:
            sequences_3di[record.id] = str(record.seq).upper()
    for seq_id in sequences_aa:
        if seq_id not in sequences_3di:
            raise SystemExit(f"AA FASTA id {seq_id} has no predicted 3Di string")

    with (tmp_dir / "aa.tsv").open("w", encoding="utf-8") as faa, \
         (tmp_dir / "3di.tsv").open("w", encoding="utf-8") as fdi, \
         (tmp_dir / "header.tsv").open("w", encoding="utf-8") as fh:
        for i, seq_id in enumerate(sequences_aa.keys(), start=1):
            idx = str(i)
            faa.write(f"{idx}\t{sequences_aa[seq_id]}\n")
            fdi.write(f"{idx}\t{sequences_3di[seq_id]}\n")
            fh.write(f"{idx}\t{seq_id}\n")


def run_tsv2db(foldseek_bin: Path, db_out: Path, tmp_dir: Path) -> None:
    cmds = [
        [str(foldseek_bin), "tsv2db", str(tmp_dir / "aa.tsv"), str(db_out), "--output-dbtype", "0"],
        [str(foldseek_bin), "tsv2db", str(tmp_dir / "3di.tsv"), f"{db_out}_ss", "--output-dbtype", "0"],
        [str(foldseek_bin), "tsv2db", str(tmp_dir / "header.tsv"), f"{db_out}_h", "--output-dbtype", "12"],
    ]
    for cmd in cmds:
        run_cmd(cmd)


def build_one(aa_fasta: Path, di_fasta: Path, db_out: Path, skip_existing: bool = True) -> Path:
    if skip_if_exists(db_out, payload=None, skip_existing=skip_existing):
        print(f"[skip] DB exists: {db_out}")
        return db_out
    require_file(FOLDSEEK_BIN, hint="Run 0_prepare_scope40.ipynb.")
    require_file(di_fasta, hint=f"Place predicted FASTA at {di_fasta}")
    db_out.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix="scope40_easy_db_") as tmp:
        tmp_dir = Path(tmp)
        build_tsvs(aa_fasta, di_fasta, tmp_dir)
        run_tsv2db(FOLDSEEK_BIN, db_out, tmp_dir)
    print(f"[ok] Foldseek DB: {db_out}")
    return db_out


out: dict[str, Path] = {
    "foldseek": FOLDSEEK_GT_DIR / "DB",
    "mmseqs": MMSEQS_GT_DIR / "DB",
}
print(f"[skip] GT foldseek: {out['foldseek']}")
print(f"[skip] GT mmseqs:   {out['mmseqs']}")

for method in selected_predicted():
    if method["engine"] != "foldseek":
        raise ValueError(f"Predicted methods must use foldseek engine: {method}")
    out[method["key"]] = build_one(
        AA_FASTA,
        AA2DI_FASTA_DIR / method["aa2di"],
        db_prefix(method["key"]),
        skip_existing=SKIP_EXISTING,
    )


## Verify


In [ ]:
print(f"{'key':12s}  exists  path")
missing = []
for key, path in out.items():
    ok = path.is_file()
    print(f"{key:12s}  {ok!s:5s}  {path}")
    if not ok:
        missing.append(key)
if missing:
    raise SystemExit(f"DB build incomplete: {missing}")
print("Next: 2a_remote_homology.ipynb (compute node) and/or 2b_translation_accuracy.ipynb")


## Cleanup


In [ ]:
cleanup_tmp(also_work_tmp=True)
print("Cleanup done. Products remain under work/ and bin/.")
